In [1]:
from bokeh.plotting import figure, output_file, show, ColumnDataSource
from bokeh.models import HoverTool

from obspy import UTCDateTime


In [2]:
pensive_info = {
    'url_syntax_s'  : 'https://avosouth.wr.usgs.gov/spectrograms/?mode=singlePlot&network=%network&subnet=%subnet&cellEndM=%cellEndM',
    'url_syntax_i'  : 'https://avosouth.wr.usgs.gov/spectrograms/%network/%subnet/%yyyy/%doy/%subnet_%yyyy%mm%dd-%HH%MM.png',
    'url_syntax_m'  : 'https://avosouth.wr.usgs.gov/spectrograms/?mode=mosaic&network=%network&subnet=%subnet&mosaicEndM=%mosaicEndM&rowSpanM=60&mosaicSpanH=3',
#     'pensive_epoch' : 27118910,
     'dateutc_epoch' : '2021/07/24 13:50:00',
}


data = {
    'network' : 'AVO',
    'subnet'  : 'Pavlof',
    'nslc'    : [],
    'filter'  : {'type':'bandpass', 'freqmin':1, 'freqmax':5}
}


# print(UTCDateTime(pensive_info['dateutc_epoch']))
# print(UTCDateTime(pensive_info['dateutc_epoch'])+10*60)

In [3]:
t = UTCDateTime(pensive_info['dateutc_epoch'])
dateutc = [t+i*10*60 for i in range(10)]
dateutc

[2021-07-24T13:50:00.000000Z,
 2021-07-24T14:00:00.000000Z,
 2021-07-24T14:10:00.000000Z,
 2021-07-24T14:20:00.000000Z,
 2021-07-24T14:30:00.000000Z,
 2021-07-24T14:40:00.000000Z,
 2021-07-24T14:50:00.000000Z,
 2021-07-24T15:00:00.000000Z,
 2021-07-24T15:10:00.000000Z,
 2021-07-24T15:20:00.000000Z]

In [4]:
#pensive_dateid = [str(int(pensive_info['pensive_epoch'] + (t2-t)/(10*60)*10 )) for t2 in dateutc]
#pensive_dateid

In [5]:
#imgurl = [ pensive_info['url_syntax_s'].replace('%network', data['network']).replace('%subnet', data['subnet']).replace('%cellEndM', cellEndM) for cellEndM in pensive_dateid]
#imgurl = [ pensive_info['url_syntax_i'].replace('%network', data['network']).replace('%subnet', data['subnet']).replace('%year', str(t.year)).replace('%mm', str(t.month)).replace('%dd', str(t.day)).replace('%doy', str(t.julday)).replace('%HH', str(t.hour)).replace('%MM', str(t.minute)) for t in dateutc]
#
#imgurl

In [6]:
def build_pensive_permalink( url_syntax, utcdatetimes ):
    #''''https://avosouth.wr.usgs.gov/spectrograms/%network/%subnet/%yyyy/%doy/%subnet_%yyyy%mm%dd-%HH%MM.png''''

    image_url = []
    for t in utcdatetimes:
        url = url_syntax
        url = url.replace('%network', data['network']).replace('%subnet', data['subnet'])
        url = url.replace('%yyyy', '{:04d}'.format(t.year) )
        url = url.replace('%doy', '{:03d}'.format(t.julday) )
        url = url.replace('%mm', '{:02d}'.format(t.month) )
        url = url.replace('%dd', '{:02d}'.format(t.day) )
        url = url.replace('%HH', '{:02d}'.format(t.hour) )
        url = url.replace('%MM', '{:02d}'.format(t.minute) )
        image_url+=[url]
        
    return image_url

def build_filename( syntax, network, subnet ):
    #"ffRSAMPensive-%network-%subnet.html"

    filename = syntax
    filename = filename.replace('%network', network)
    filename = filename.replace('%subnet', subnet)
    
    return filename

In [7]:
imgurl = build_pensive_permalink( pensive_info['url_syntax_i'], dateutc )
imgurl

['https://avosouth.wr.usgs.gov/spectrograms/AVO/Pavlof/2021/205/Pavlof_20210724-1350.png',
 'https://avosouth.wr.usgs.gov/spectrograms/AVO/Pavlof/2021/205/Pavlof_20210724-1400.png',
 'https://avosouth.wr.usgs.gov/spectrograms/AVO/Pavlof/2021/205/Pavlof_20210724-1410.png',
 'https://avosouth.wr.usgs.gov/spectrograms/AVO/Pavlof/2021/205/Pavlof_20210724-1420.png',
 'https://avosouth.wr.usgs.gov/spectrograms/AVO/Pavlof/2021/205/Pavlof_20210724-1430.png',
 'https://avosouth.wr.usgs.gov/spectrograms/AVO/Pavlof/2021/205/Pavlof_20210724-1440.png',
 'https://avosouth.wr.usgs.gov/spectrograms/AVO/Pavlof/2021/205/Pavlof_20210724-1450.png',
 'https://avosouth.wr.usgs.gov/spectrograms/AVO/Pavlof/2021/205/Pavlof_20210724-1500.png',
 'https://avosouth.wr.usgs.gov/spectrograms/AVO/Pavlof/2021/205/Pavlof_20210724-1510.png',
 'https://avosouth.wr.usgs.gov/spectrograms/AVO/Pavlof/2021/205/Pavlof_20210724-1520.png']

In [8]:
output_file(build_filename("ffRSAMPensive-%network-%subnet.html", data['network'], data['subnet']))

# stub data
source = ColumnDataSource(
        data=dict(
            x =[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
            y =[1, 2, 3, 4, 3, 3, 1, 1, 6, 15],
            y0=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            desc=['A', 'b', 'C', 'd', 'E'],
            imgs=imgurl
  
        )
    )

# margin: top, left, bottom, right
scale=0.55; width=576*scale; height=756*scale # 576 × 756 is dimension of original image
tooltipstr = """
        <div>
            <div>
                <img
                    src="@imgs" height="%height" alt="@imgs" width="%width"
                    style="float: center; margin: 0px 0px 0px 0px;"
                    border="2"
                ></img>
            </div>
        </div>
        """
tooltipstr = tooltipstr.replace('%height', str(height)).replace('%width', str(width))

hover = HoverTool(
    tooltips=tooltipstr,
    mode='vline',
    attachment='below', # vertical, horizontal, 'below', ...
    point_policy='snap_to_data', # ['snap_to_data'], 'follow_mouse' 'none'
    anchor='center', # not sure what effect this has
    )


p = figure(plot_width=1200, plot_height=200, tools=[hover],
           title="ffRSAM")

p.circle('x', 'y', size=10, source=source)
#p.circle('x', 'y0', size=20, source=source)


show(p)

In [9]:
print(UTCDateTime(pensive_info['dateutc_epoch']))
print(UTCDateTime(pensive_info['dateutc_epoch'])+10*60)

2021-07-24T13:50:00.000000Z
2021-07-24T14:00:00.000000Z


In [10]:
t = UTCDateTime(pensive_info['dateutc_epoch'])
dateutc = [t+i*10*60 for i in range(10)]
dateutc

[2021-07-24T13:50:00.000000Z,
 2021-07-24T14:00:00.000000Z,
 2021-07-24T14:10:00.000000Z,
 2021-07-24T14:20:00.000000Z,
 2021-07-24T14:30:00.000000Z,
 2021-07-24T14:40:00.000000Z,
 2021-07-24T14:50:00.000000Z,
 2021-07-24T15:00:00.000000Z,
 2021-07-24T15:10:00.000000Z,
 2021-07-24T15:20:00.000000Z]

In [11]:
t2 = UTCDateTime(pensive_info['dateutc_epoch'])+1*10*60
t2

2021-07-24T14:00:00.000000Z

In [24]:
mm= 19; mm-(mm%10)

10